# P12A — Generator panel inspeksi jembatan sintetis

Notebook ini membangkitkan panel sintesis 18 jembatan × 36 bulan. Setiap baris adalah satu pengamatan pada satu aset dan bulan. Fitur menggabungkan lingkungan, respons sensor, serta indikator dari inspeksi visual. Target adalah indeks kondisi 3 bulan kemudian. Seluruh koordinat, aset, dan angka dibuat dari proses sintesis untuk latihan; bukan data lokasi atau struktur nyata. Jalankan dari root repository.


In [ ]:
# Folder kerja: jalankan dari folder mana pun. Sel ini pindah ke akar repositori agar jalur data relatif (praktikum/...) benar.
import os
from pathlib import Path
_akar = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "praktikum").is_dir() and (p / "notebooks").is_dir()), None)
if _akar is not None:
    os.chdir(_akar)
else:
    print("Akar repositori tidak ditemukan. Buka notebook dari dalam folder repositori hasil clone.")

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd

SEED = 12012
rng = np.random.default_rng(SEED)
N_BRIDGES = 18
N_MONTHS = 36
rows = []

# Koordinat proyeksi sintetis dalam km; tidak menunjuk aset nyata.
site_xy = rng.uniform([0, 0], [32, 24], size=(N_BRIDGES, 2))
site_effect = rng.normal(0, 3.0, N_BRIDGES)
bridge_ages = rng.integers(8, 56, N_BRIDGES)
span_lengths = rng.uniform(18, 72, N_BRIDGES)
bridge_types = rng.choice(['girder', 'slab', 'truss'], N_BRIDGES, p=[.56,.28,.16])

for b in range(N_BRIDGES):
    base = rng.uniform(12, 34) + site_effect[b]
    growth = rng.uniform(.12, .42)
    phase = rng.uniform(0, 2*np.pi)
    for m in range(N_MONTHS):
        rain = max(0, 185 + 95*np.sin(2*np.pi*m/12 + phase) + rng.normal(0, 45))
        temp = 27 + 3.5*np.sin(2*np.pi*(m+2)/12) + rng.normal(0, 1.1)
        traffic = np.clip(rng.normal(0.58 + .12*np.sin(2*np.pi*m/12), .12), .15, 1.0)
        current = np.clip(base + growth*m + .035*rain + 5*traffic + rng.normal(0, 2.0), 0, 100)
        crack = np.clip(.035*current + rng.normal(0, .34), 0, 8)
        corrosion = np.clip(.045*current + rng.normal(0, .55), 0, 5)
        vibration = max(.02, .42 + .012*current + .06*traffic + rng.normal(0, .09))
        displacement = max(0, .18 + .018*current + .06*traffic + rng.normal(0, .13))
        sensor_cov = np.clip(rng.normal(93, 8), 48, 100)
        photo_offset = int(rng.integers(-12, 13))
        confidence = np.clip(rng.beta(9, 2), .35, 1.0)
        # Degradation response over the next three months: DGP is instructional.
        future = np.clip(current + 3*growth + .012*rain + .7*traffic + rng.normal(0, 2.6), 0, 100)
        # Observation date begins in 2020; entirely synthetic.
        date = pd.Timestamp('2020-01-01') + pd.DateOffset(months=m)
        image_available = rng.random() > .12
        sensor_available = rng.random() > .07
        rows.append({
            'bridge_id': f'SYN-{b+1:02d}', 'observation_month': date.strftime('%Y-%m'),
            'month_index': m, 'easting_km_synth': site_xy[b,0], 'northing_km_synth': site_xy[b,1],
            'bridge_type': bridge_types[b], 'bridge_age_years': bridge_ages[b], 'span_m': span_lengths[b],
            'rain_30d_mm': rain, 'temperature_C': temp, 'traffic_index': traffic,
            'vibration_rms': vibration if sensor_available else np.nan,
            'displacement_mm': displacement if sensor_available else np.nan,
            'sensor_coverage_pct': sensor_cov if sensor_available else np.nan,
            'crack_ratio_pct': crack if image_available else np.nan,
            'corrosion_visual_score': corrosion if image_available else np.nan,
            'photo_offset_days': photo_offset if image_available else np.nan,
            'inspection_confidence': confidence if image_available else np.nan,
            'condition_current_synth': current, 'condition_next_3m_synth': future})

df = pd.DataFrame(rows)
assert len(df) == N_BRIDGES*N_MONTHS
assert not df.duplicated(['bridge_id','observation_month']).any()


In [2]:
outdir = Path('praktikum/bab12')
outdir.mkdir(parents=True, exist_ok=True)
csv_path = outdir/'bridge_panel_synthetic.csv'
tab_path = outdir/'bridge_panel_orange.tab'
df.to_csv(csv_path, index=False)
# Orange .tab descriptor: names, types, then variable roles.
roles=[]
for col in df.columns:
    roles.append('meta' if col in ['bridge_id','observation_month'] else ('class' if col=='condition_next_3m_synth' else ''))
with tab_path.open('w', encoding='utf-8', newline='') as f:
    f.write('\t'.join(df.columns)+'\n')
    f.write('\t'.join('string' if c in ['bridge_id','observation_month'] else ('discrete' if c=='bridge_type' else 'continuous') for c in df.columns)+'\n')
    f.write('\t'.join('meta' if c in ['bridge_id','observation_month'] else ('class' if c=='condition_next_3m_synth' else '') for c in df.columns)+'\n')
    df.to_csv(f, sep='\t', index=False, header=False, na_rep='?')
print(f'rows={len(df)} | bridges={df.bridge_id.nunique()} | months={df.observation_month.nunique()} | seed={SEED}')
print('missing sensor rows:', int(df.vibration_rms.isna().sum()), '| missing visual rows:', int(df.crack_ratio_pct.isna().sum()))
print('target range:', round(df.condition_next_3m_synth.min(),2), 'to', round(df.condition_next_3m_synth.max(),2))
print(df.head(4).round(2).to_string(index=False))
print('saved:', csv_path, tab_path)


rows=648 | bridges=18 | months=36 | seed=12012
missing sensor rows: 54 | missing visual rows: 73
target range: 15.6 to 66.44
bridge_id observation_month  month_index  easting_km_synth  northing_km_synth bridge_type  bridge_age_years  span_m  rain_30d_mm  temperature_C  traffic_index  vibration_rms  displacement_mm  sensor_coverage_pct  crack_ratio_pct  corrosion_visual_score  photo_offset_days  inspection_confidence  condition_current_synth  condition_next_3m_synth
   SYN-01           2020-01            0             26.66              21.35        slab                50   20.51       205.82          30.64           0.34            NaN              NaN                  NaN             0.85                    1.20                1.0                   0.83                    16.70                    21.86
   SYN-01           2020-02            1             26.66              21.35        slab                50   20.51        64.13          30.51           0.44           0.60            